#### Explanation
The H0(null) and H1 (alternative) hypotheses are opposing, mutually exclusive statements used in statistics to determine if sample data supports a specific theory

H0 asserts no effect or no difference (business as usual), while H1 claims a significant effect or difference exists\
α - significance level

Types of Errors:
Type I Error (False Positive): Rejecting H0 when it is actually true.\
Type II Error (False Negative): Failing to reject H0 when it is actually false H1 and is true. 

Now that we have done EDA on the dataset, we are moving to AB testing part. 

Framing experiment:
"The bank tested two contact strategies: cellular (mobile) vs telephone (landline).\
We want to know if contact channel significantly affects subscription conversion rate."

H0: Contact channel has no effect on conversion rate\
H1: Contact channel affects conversion rate



<img src="img.png" width="500">


In [11]:
import pandas as pd
import math
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sys
from stats import z_test, check_srm, sample_size_calculator, segment_test

In [23]:
data = pd.read_csv("/Users/tolkyn.zhagipar/Desktop/pet-projects/fintech-ab-test/notebooks/cleaned_data.csv")
data.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y,y_binary,is_returning
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,0,0,0
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,0,0,0
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,0,0,0
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,0,0,0
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,0,0,0


In [57]:
sys.path.append('../src')

#test cellular vs telephone conversion
cellular = data[data['contact'] == 'cellular']
telephone = data[data['contact'] == 'telephone']

In [59]:
# Are the two groups roughly equal in size? Use SRM 

srm_result = check_srm(
    expected_split = [0.5, 0.5],
    actual_counts = [len(cellular), len(telephone)]
)

print(srm_result)

{'expected_counts': [16096, 16096], 'actual_counts': [29285, 2906], 'chi2_statistic': 21616.3412, 'p_value': 0.0, 'srm_detected': True}


Groups are heavily imbalanced: cellular (29,285) vs telephone (2,906).
SRM(Sample Ratio Mismatch) is detected but expected — this is observational data where contact 
channel was chosen by the bank, not randomly assigned.

Implication: results should be interpreted with caution. The cellular 
group may differ from telephone group in ways beyond just contact channel 

In [63]:
result = z_test(
    control_conversions=telephone['y'].sum(), 
    control_n=len(telephone), 
    treatment_conversions=cellular['y'].sum(), 
    treatment_n=len(cellular)
)

print(result)

{'control_rate': 0.1342, 'treatment_rate': 0.1492, 'lift': 0.015, 'z_score': 2.1706, 'p_value': 0.03, 'confidence_interval_95': (0.0019, 0.028)}


#### Result: Reject H0 (p=0.03, z=2.17)

Cellular outperforms telephone by 1.5pp (14.92% vs 13.42%).
The effect is statistically significant but modest in practical terms.
The 95% CI (0.19%, 2.8%) suggests the true lift could be as small 
as 0.19pp — worth validating with a properly controlled experiment 
before making channel strategy decisions.

Caution: given the severe group imbalance (91% vs 9%), this result 
should be treated as directional, not conclusive.

In [91]:
# --- Practical Significance ---

# Group sizes
n_cellular = len(cellular)
n_telephone = len(telephone)

# Conversion rates from z_test result
cr_telephone = 0.1342
cr_cellular = 0.1492
lift = cr_cellular - cr_telephone  # 1.5pp

# Additional conversions if we switch everyone to cellular
additional_conversions = lift * n_telephone
print(f"Additional conversions if telephone users switched to cellular: {additional_conversions:.0f}")

Additional conversions if telephone users switched to cellular: 44


In [93]:
# Revenue estimate — assume average deposit value
avg_deposit = 5000  # USD, state this is an assumption
bank_margin = 0.02  # banks typically earn ~2% on deposits

revenue_impact = additional_conversions * avg_deposit * bank_margin
print(f"Estimated revenue impact: ${revenue_impact:,.0f}")

# Minimum Detectable Effect check
mde = 0.02  # we defined 2pp as the minimum lift worth acting on
print(f"\nPractical significance threshold (MDE): {mde:.1%}")
print(f"Observed lift: {lift:.1%}")
print(f"Practically significant: {lift >= mde}")

Estimated revenue impact: $4,359

Practical significance threshold (MDE): 2.0%
Observed lift: 1.5%
Practically significant: False


Statistical significance alone doesn't justify a business decision.
- Observed lift: +1.5pp (below our 2pp MDE threshold)
- Estimated additional conversions: ~44 clients
- Estimated revenue impact: ~$4,400 (assuming $5K avg deposit, 2% margin)

Conclusion: The effect is statistically significant but NOT practically 
significant by our pre-defined MDE of 2pp. The revenue impact is modest 
and the severe group imbalance (91/9 split) weakens confidence.\
Recommendation: Run a properly controlled experiment with equal group 
sizes before committing to a cellular-first strategy.